# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [3]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage, SystemMessage
import tiktoken
from IPython.display import Markdown, display
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

In [12]:
load_dotenv()

# Check if API keys are set
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY not found in environment. Please create a .env file with TAVILY_API_KEY=your_key_here")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment. Please add OPENAI_API_KEY to your .env file")

tavily_search = TavilySearchResults(max_results=5)

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4")

# Safe Tavily wrapper with better error handling
def safe_search(query: str) -> str:
    try:
        # Shorten the query if it's too long (Tavily has limits)
        search_query = query[:400] if len(query) > 400 else query
        
        result = tavily_search.invoke({"query": search_query})
        
        # Handle different result formats
        if isinstance(result, list):
            # Extract content from list of results
            result_text = "\n\n".join([
                f"Source: {r.get('url', 'N/A')}\n{r.get('content', '')}" 
                for r in result
            ])
        elif isinstance(result, dict):
            result_text = result.get("content", "") or str(result)
        else:
            result_text = str(result)
        
        # Trim to avoid token limits
        tokens = encoding.encode(result_text)
        trimmed = encoding.decode(tokens[:3500])
        return trimmed
        
    except Exception as e:
        return f"Search error: {str(e)}. Please try a shorter, simpler query."


# Simple agent function
def run_agent(prompt: str) -> str:
    """Simple agent that searches and generates response"""
    # Extract just the core search query from the prompt
    # This helps avoid sending formatting instructions to Tavily
    search_query = prompt.split("Please provide")[0].strip()
    if not search_query:
        search_query = prompt[:200]  # Fallback to first 200 chars
    
    # Do the search with the simplified query
    search_results = safe_search(search_query)
    
    # Then, ask LLM to format the results
    messages = [
        SystemMessage(content="You are a helpful assistant. Use the provided search results to answer the user's question in a well-formatted Markdown response."),
        HumanMessage(content=f"Search Results:\n{search_results}\n\nUser Question: {prompt}\n\nPlease provide a comprehensive answer in Markdown format.")
    ]
    
    response = llm.invoke(messages)
    return response.content

# Create a simple wrapper object for compatibility
class SimpleAgent:
    def run(self, prompt):
        return run_agent(prompt)
    
    def invoke(self, inputs):
        return {"output": run_agent(inputs["input"])}

agent = SimpleAgent()

# Quick test to verify Tavily API is working
try:
    test_result = tavily_search.invoke({"query": "AI news"})
    print("✅ Agent setup complete! Tavily API is working.")
except Exception as e:
    print(f"⚠️ Warning: Tavily API test failed: {e}")
    print("Please check your TAVILY_API_KEY in the .env file.")

✅ Agent setup complete! Tavily API is working.


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [7]:
# Your prompt here
prompt_1 = """Search for the latest advancements in generative AI applied to healthcare in 2025. 

Please provide a comprehensive summary including:
- Key breakthroughs and innovations
- Specific applications (diagnosis, drug discovery, patient care, etc.)
- Leading companies or research institutions
- Impact on the healthcare industry

Format your response in clear Markdown with headers, bullet points, and sources where available."""

# Run it
response_1 = agent.run(prompt_1)

In [8]:
display(Markdown(response_1))


I'm sorry, but I was unable to retrieve the specific information you requested due to a technical error. However, I can provide a general overview of the potential advancements in generative AI applied to healthcare based on current trends and predictions. Please note that this information may not be entirely accurate as it is speculative and based on current trends, not actual data from 2025.

# Generative AI in Healthcare: Potential Advancements

## Key Breakthroughs and Innovations

- **Improved Diagnosis**: Generative AI models could potentially be trained to generate synthetic medical images or patient data, which could be used to improve the accuracy of disease diagnosis.
- **Drug Discovery**: Generative AI could be used to generate new potential drug molecules, speeding up the drug discovery process.
- **Personalized Treatment Plans**: Generative AI could potentially be used to generate personalized treatment plans based on a patient's unique medical history and genetic makeup.

## Specific Applications

- **Diagnosis**: Generative AI could be used to improve the accuracy of disease diagnosis by generating synthetic medical images or patient data.
- **Drug Discovery**: Generative AI could be used to generate new potential drug molecules, speeding up the drug discovery process.
- **Patient Care**: Generative AI could potentially be used to generate personalized treatment plans based on a patient's unique medical history and genetic makeup.

## Leading Companies or Research Institutions

- **Google's DeepMind**: DeepMind has been at the forefront of AI research and has made significant contributions to the field of generative AI.
- **OpenAI**: OpenAI is another leading research institution in the field of AI and has made significant advancements in generative AI.

## Impact on the Healthcare Industry

- **Improved Patient Outcomes**: By improving the accuracy of disease diagnosis and personalizing treatment plans, generative AI could potentially lead to improved patient outcomes.
- **Increased Efficiency**: By speeding up the drug discovery process, generative AI could potentially increase efficiency in the healthcare industry.
- **Cost Savings**: By improving patient outcomes and increasing efficiency, generative AI could potentially lead to significant cost savings in the healthcare industry.

Please note that this information is speculative and based on current trends, not actual data from 2025.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [13]:
# Your prompt here
prompt_2 = """Search for the top emerging AI startups in 2025 and their innovations.

Please provide a structured summary including:
- Company name and location
- Main product or service offering
- Key innovations or breakthrough technology
- Funding information or valuation (if available)
- Target market or industry
- Recent news or achievements
- Source links

Format as Markdown with clear sections or a table for easy comparison."""
# Run it
response_2 = agent.run(prompt_2)

In [14]:
display(Markdown(response_2))


Here are some of the top emerging AI startups in 2025 and their innovations:

1. **Anysphere (Cursor)**
   - Location: Not specified
   - Main Product/Service: Helps engineers write and edit code
   - Key Innovations: AI-powered code editing
   - Funding/Valuation: Valued at $2.5 billion with at least $100 million in annualized revenue
   - Target Market: Engineering
   - Recent News: Included in Forbes' seventh annual AI 50 list
   - Source: [Forbes](https://www.forbes.com/lists/ai50/)

2. **Speak**
   - Location: Not specified
   - Main Product/Service: AI language tutor app
   - Key Innovations: AI-powered language learning
   - Funding/Valuation: Valued at $1 billion
   - Target Market: Education
   - Recent News: Used by some 10 million people to learn English and Spanish
   - Source: [Forbes](https://www.forbes.com/lists/ai50/)

3. **OpenEvidence**
   - Location: Massachusetts, USA
   - Main Product/Service: AI-powered search platform that summarizes medical information for doctors
   - Key Innovations: AI-powered medical information summarization
   - Funding/Valuation: Not specified
   - Target Market: Healthcare
   - Recent News: Included in Forbes' seventh annual AI 50 list
   - Source: [Forbes](https://www.forbes.com/lists/ai50/)

4. **QuantumLeap**
   - Location: UK
   - Main Product/Service: AI and machine learning for computing
   - Key Innovations: Generative AI and advanced algorithms for computing
   - Funding/Valuation: Not specified
   - Target Market: Computing
   - Recent News: Changing the way we think about computing with AI
   - Source: [Accountancy Cloud](https://accountancycloud.com/blogs/10-hottest-ai-startups-to-watch-2025)

5. **Synthetix.AI**
   - Location: UK
   - Main Product/Service: Synthetic data solutions
   - Key Innovations: Generative AI and machine learning for managing unstructured data
   - Funding/Valuation: Not specified
   - Target Market: Data management
   - Recent News: Working with the University of Oxford to advance AI research
   - Source: [Accountancy Cloud](https://accountancycloud.com/blogs/10-hottest-ai-startups-to-watch-2025)

6. **AI HealthSolutions**
   - Location: UK
   - Main Product/Service: AI technology for healthcare
   - Key Innovations: AI for more accurate patient care
   - Funding/Valuation: Not specified
   - Target Market: Healthcare
   - Recent News: Working with the NHS and the University of Oxford to improve health services
   - Source: [Accountancy Cloud](https://accountancycloud.com/blogs/10-hottest-ai-startups-to-watch-2025)

7. **Chai Discovery**
   - Location: USA
   - Main Product/Service: AI models for biotech and drug discovery
   - Key Innovations: AI-powered drug discovery
   - Funding/Valuation: Raised $130 million in Series B round, valued at $1.2 billion
   - Target Market: Biotech and drug discovery
   - Recent News: Announced a $130 million Series B round
   - Source: [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

8. **Fal**
   - Location: USA
   - Main Product/Service: Generative media platform
   - Key Innovations: AI-powered generative media
   - Funding/Valuation: Raised $140 million in Series D round, valued at more than $4.5 billion
   - Target Market: Media
   - Recent News: Announced its third funding round of 2025
   - Source: [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

9. **Unconventional AI**
   - Location: USA
   - Main Product/Service: Rethinking the foundation of computers in the age of AI
   - Key Innovations: Novel approach to computing in the AI era
   - Funding/Valuation: Raised $475 million in seed round, valued at nearly $4.5 billion
   - Target Market: Computing
   - Recent News: Announced a $475 million seed round
   - Source: [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

10. **Cognition AI**
    - Location: USA
    - Main Product/Service: Creator of vibe-coding agent Devin
    - Key Innovations: AI-powered vibe-coding
    - Funding/Valuation: Raised $400 million in Series C round, valued at $10.2 billion
    - Target Market: Not specified
    - Recent News: Raised a $400 million Series C round
    - Source: [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [15]:
# Your prompt here
prompt_3 = """Compare OpenAI's ChatGPT Team and Anthropic's Claude Pro 2025.

Please provide a comprehensive comparison including:
- Key features and capabilities of each product
- Pricing and subscription tiers
- Performance benchmarks or reviews
- Target users or use cases
- Pros and cons of each
- Recent updates or improvements

Format your response in clear Markdown with headers, comparison table, and bullet points."""

# Run it
response_3 = agent.run(prompt_3)

In [16]:
display(Markdown(response_3))


# OpenAI's ChatGPT Team vs Anthropic's Claude Pro (2025)

## Key Features and Capabilities

### ChatGPT Team

- **Models**: ChatGPT Team uses the GPT-4o model.
- **User Experience**: ChatGPT comes with plug-ins, built-in tools, memory, and Microsoft 365 links.
- **Performance**: GPT-4.1 scores highest on general knowledge tests.
- **Updates**: OpenAI rolls out new features quickly.

### Claude Pro

- **Models**: Claude Pro uses the Sonnet model.
- **User Experience**: Claude answers faster and can keep very long conversations in one go. It can also call external APIs and built-in tools.
- **Performance**: Claude Opus is best at step-by-step coding tasks.
- **Updates**: Anthropic moves slowly and puts safety first, with fewer but bigger jumps (Claude 3 → 3.5).

## Pricing and Subscription Tiers

| Product | Monthly Price | Included Models | Notes |
| --- | --- | --- | --- |
| ChatGPT Team | $25–$30/user | GPT‑4o | Shared workspaces |
| Claude Pro | $20 | Sonnet | 5× usage vs free |

## Performance Benchmarks or Reviews

- ChatGPT scores highest on general knowledge tests.
- Claude Opus is best at step-by-step coding tasks.

## Target Users or Use Cases

- ChatGPT is suitable for individuals to enterprises, providing general AI tools.
- Claude's pricing is more tailored for professional use, with tiers offering greater access to powerful models and higher usage caps for specialized tasks.

## Pros and Cons

### ChatGPT Team

**Pros**:

- Comes with plug-ins, built-in tools, memory, and Microsoft 365 links.
- Scores highest on general knowledge tests.

**Cons**:

- Rolls out new features quickly, which may lead to instability.

### Claude Pro

**Pros**:

- Answers faster and can keep very long conversations in one go.
- Can call external APIs and built-in tools.
- Best at step-by-step coding tasks.

**Cons**:

- Moves slowly and puts safety first, which may delay the release of new features.

## Recent Updates or Improvements

- ChatGPT: OpenAI rolls out new features quickly.
- Claude: Notable 2025 upgrade — Tool use & Agents: Claude can now call external APIs and built-in tools, making it competitive with ChatGPT’s plug-ins and Code Interpreter.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


In [17]:
# Your prompt here
prompt_4 = """Find the best AI and machine learning tools for learning physics and mathematics in 2025-2026.

Please research and provide:
- **AI Tutoring Platforms**: Tools with step-by-step problem solving for physics and maths
- **Specialized Tools**: 
  - Physics simulations and visualization tools powered by AI
  - Mathematics symbolic computation and algebra helpers
  - Problem solvers with explanations
- **Features to highlight**:
  - Interactive explanations
  - Personalized learning paths
  - Real-time feedback
  - Physics equation visualization
  - Calculus and algebra assistance
- **Popular Tools**: ChatGPT, Claude, Wolfram Alpha, CAS systems, Photomath alternatives
- **Educational Impact**: How these AI tools improve learning outcomes
- **Free vs Paid**: Cost and accessibility options

Format as Markdown with sections for each tool, pricing, and use cases for physics/maths students."""
# Run it
response_4 = agent.run(prompt_4)

In [18]:
display(Markdown(response_4))

# Best AI and Machine Learning Tools for Learning Physics and Mathematics in 2025-2026

## AI Tutoring Platforms

### Wolfram Alpha Pro
Wolfram Alpha Pro is a powerful tool for physics and mathematics, excelling in symbolic computation, differential equations, and data analysis. It features tensor support, physics notation, animated plots, and an extensive knowledge base. Ideal for theory problems, homework verification, and data analysis, it's a great tool for students and researchers alike. 

**Price**: [Visit website for pricing](https://www.wolframalpha.com/pro/)

### Symbolab
Symbolab offers guided steps for algebra, calculus, and physics contexts, including vector fields and energy concepts. It supports plain input with LaTeX, phase-space graphs, and energy curves. It's ideal for homework checks, exam prep, and quick problem validation.

**Price**: [Visit website for pricing](https://www.symbolab.com/)

### Microsoft Math Solver
Microsoft Math Solver blends AI assistance with step-by-step explanations for algebra, calculus, and physics-themed problems. It covers dynamics, kinematics, electromagnetism, and thermodynamics topics with units and dimensional checks. It's ideal for quick checks, homework help, and test prep.

**Price**: Free

### AI Tutor
AI Tutor offers step-by-step support on challenging physics topics and algebra, helping students stay prepared for assignments and exams. 

**Price**: [Visit website for pricing](https://www.pearson.com/channels/ai-tutor)

### MathGPT
MathGPT provides step-by-step math and physics help, teaching you how to solve problems. It's the most accurate and is ideal for students needing detailed explanations.

**Price**: $14.99/month

## Specialized Tools

### GeoGebra
GeoGebra offers dynamic geometry with calculus and physics visuals, vector representation, and 2D/3D plots. It features live graphs, motion constraints, kinematic plots, and sliders. It's ideal for mechanics intuition, classroom demonstrations, and labs.

**Price**: Free

### Desmos
Desmos is a fast graphing tool with trajectory plots, parametric/polar support, and visuals. It features interactive sliders and unit labeling. It's ideal for intuition building, rapid scenario testing, and demonstrations.

**Price**: Free

### Maple
Maple offers symbolic and numeric solvers, tensor algebra, Lagrangian/Hamiltonian methods, and differential equations. It features a physics library, reproducible notebooks, and phase portraits. It's ideal for graduate-level work, research, and complex models.

**Price**: [Visit website for pricing](https://www.maplesoft.com/products/Maple/)

## Features to Highlight

These tools offer interactive explanations, personalized learning paths, real-time feedback, physics equation visualization, and calculus and algebra assistance. They improve learning outcomes by providing detailed, step-by-step solutions, promoting understanding of complex concepts, and offering immediate feedback.

## Free vs Paid

While some tools like Microsoft Math Solver, GeoGebra, and Desmos are free, others like Wolfram Alpha Pro, Symbolab, AI Tutor, and MathGPT offer more advanced features and require a subscription. Prices vary, so it's best to visit the respective websites for the most accurate information.